# Inference Engine — Foundations

> **Section 01 — Topic 07 — foundations**

**Prerequisites:** Basic understanding of transformer architecture (Topics 01-06)

**What you'll learn:**
- How autoregressive generation actually works, token by token
- Every major decoding strategy: greedy, beam search, sampling variants
- KV caching and why it makes generation dramatically faster
- Stopping criteria and structured output generation

**What you'll build:**
- A complete decode loop from scratch with all sampling strategies
- A KV cache implementation with benchmarks showing the speedup
- A constrained JSON decoder

## Setup

In [ ]:
!pip install -q torch transformers matplotlib numpy

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load GPT-2 small — perfect for understanding inference mechanics
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

tokenizer.pad_token = tokenizer.eos_token
print(f"Model: {model_name} ({sum(p.numel() for p in model.parameters()):,} params)")

---
## 1. Autoregressive Generation — The Decode Loop from Scratch

Every LLM generates text the same fundamental way: **one token at a time**. The model takes in a sequence of tokens, produces a probability distribution over the entire vocabulary for the *next* token, we select one token from that distribution, append it, and repeat.

This is called **autoregressive generation** because each prediction depends on ("regresses on") the model's own previous outputs.

```
Input:  [The, cat, sat]
Model → P(next | The, cat, sat) → selects "on"
Input:  [The, cat, sat, on]
Model → P(next | The, cat, sat, on) → selects "the"
Input:  [The, cat, sat, on, the]
Model → P(next | The, cat, sat, on, the) → selects "mat"
```

Let's implement this from scratch to see every step.

In [ ]:
def autoregressive_generate(model, tokenizer, prompt, max_new_tokens=20):
    """The simplest possible decode loop — greedy, no caching."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    
    steps = []  # Track each step for visualization
    
    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            # outputs.logits shape: [batch, seq_len, vocab_size]
            # We only care about the LAST position's predictions
            next_token_logits = outputs.logits[:, -1, :]  # [batch, vocab_size]
        
        # Convert logits to probabilities
        probs = F.softmax(next_token_logits, dim=-1)
        
        # Greedy: pick the highest probability token
        next_token = torch.argmax(probs, dim=-1, keepdim=True)
        
        # Record step info
        top_prob = probs[0, next_token[0, 0]].item()
        token_text = tokenizer.decode(next_token[0])
        steps.append({
            'step': step,
            'token_id': next_token[0, 0].item(),
            'token_text': token_text,
            'probability': top_prob,
        })
        
        # Append to sequence
        generated = torch.cat([generated, next_token], dim=-1)
        
        # Stop if we hit EOS
        if next_token[0, 0].item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(generated[0]), steps

# Run it
prompt = "The future of artificial intelligence is"
text, steps = autoregressive_generate(model, tokenizer, prompt, max_new_tokens=30)
print(f"Prompt: {prompt}")
print(f"Generated: {text}\n")
print("Step-by-step:")
for s in steps:
    print(f"  Step {s['step']:2d}: '{s['token_text']}' (p={s['probability']:.4f})")

In [ ]:
# Visualize the generation process — probability at each step
fig, ax = plt.subplots(figsize=(14, 4))

probs = [s['probability'] for s in steps]
labels = [s['token_text'].strip() or '\\n' for s in steps]

colors = ['#2ecc71' if p > 0.5 else '#f39c12' if p > 0.1 else '#e74c3c' for p in probs]
bars = ax.bar(range(len(probs)), probs, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Probability of chosen token')
ax.set_title('Autoregressive Generation — Token Probabilities at Each Step')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, label='50% confidence')
ax.legend()
plt.tight_layout()
plt.show()

print("Green = model is confident | Orange = moderate | Red = uncertain")

### What's happening under the hood

At every step, the model runs a **full forward pass** through all transformer layers. The output at position $t$ is a vector of logits $z_t \in \mathbb{R}^{|V|}$ over the entire vocabulary. We convert these to probabilities:

$$P(x_t | x_{<t}) = \text{softmax}(z_t)$$

The overall probability of a generated sequence is:

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t | x_{<t})$$

This is key: **greedy decoding does NOT find the most probable sequence** — it finds the sequence of individually most-probable tokens. These can be very different things.

---
## 2. Decoding Strategies — Greedy and Beam Search

### 2.1 Greedy Decoding

We already implemented greedy decoding above: at each step, pick $\arg\max P(x_t | x_{<t})$.

**Pros:** Fast, deterministic, simple.  
**Cons:** Gets stuck in repetitive loops, misses better sequences that start with lower-probability tokens.

### 2.2 Beam Search

Beam search maintains $k$ candidate sequences ("beams") at each step, expanding each by all possible next tokens, then keeping only the top $k$ by total sequence probability. This explores more of the search space without full exponential blowup.

In [ ]:
def beam_search(model, tokenizer, prompt, beam_width=4, max_new_tokens=20):
    """
    Beam search implementation from scratch.
    
    Each beam is (token_ids, cumulative_log_prob).
    At each step we expand all beams and keep top-k.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    # Initialize beams: list of (sequence_tensor, cumulative_log_prob)
    beams = [(input_ids, 0.0)]
    completed = []
    
    for step in range(max_new_tokens):
        all_candidates = []
        
        for seq, score in beams:
            with torch.no_grad():
                outputs = model(seq)
                logits = outputs.logits[:, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)
            
            # Get top-k tokens for this beam
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_width, dim=-1)
            
            for i in range(beam_width):
                token_id = topk_ids[0, i].unsqueeze(0).unsqueeze(0)
                new_seq = torch.cat([seq, token_id], dim=-1)
                new_score = score + topk_log_probs[0, i].item()
                
                if token_id.item() == tokenizer.eos_token_id:
                    completed.append((new_seq, new_score))
                else:
                    all_candidates.append((new_seq, new_score))
        
        if not all_candidates:
            break
        
        # Keep top beam_width candidates
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]
    
    # Add remaining beams to completed
    completed.extend(beams)
    completed.sort(key=lambda x: x[1], reverse=True)
    
    results = []
    for seq, score in completed[:beam_width]:
        text = tokenizer.decode(seq[0], skip_special_tokens=True)
        # Length-normalize the score
        prompt_len = input_ids.shape[1]
        gen_len = seq.shape[1] - prompt_len
        norm_score = score / gen_len if gen_len > 0 else score
        results.append((text, score, norm_score))
    
    return results

prompt = "The key to building great software is"
results = beam_search(model, tokenizer, prompt, beam_width=4, max_new_tokens=25)

print(f"Prompt: {prompt}\n")
print("Beam search results (sorted by normalized score):")
for i, (text, raw_score, norm_score) in enumerate(results):
    generated_part = text[len(prompt):]
    print(f"\n  Beam {i+1} (score={norm_score:.3f}):")
    print(f"  ...{generated_part}")

### Beam search produces more coherent text but is repetitive

This is a well-known problem: beam search tends to produce **degenerate, repetitive** text because high-probability sequences often loop. This motivated the move toward **sampling-based** methods for open-ended generation.

**When to use beam search:**
- Translation (where there's one "right" answer)
- Summarization
- Code generation (where correctness matters more than diversity)

**When NOT to use beam search:**
- Creative writing, dialogue, brainstorming

---
## 3. Sampling — Temperature, Top-k, Top-p (Nucleus)

Instead of always picking the most likely token, we **sample** from the distribution. This introduces randomness and creativity but needs to be controlled to avoid nonsense.

### 3.1 Temperature Scaling

Temperature $\tau$ reshapes the probability distribution before sampling:

$$P(x_t) = \frac{\exp(z_t / \tau)}{\sum_j \exp(z_j / \tau)}$$

- $\tau = 1.0$: Original distribution
- $\tau < 1.0$: **Sharper** — more confident, less random
- $\tau > 1.0$: **Flatter** — more uniform, more random
- $\tau \to 0$: Becomes greedy decoding
- $\tau \to \infty$: Becomes uniform random

In [ ]:
def get_next_token_probs(model, input_ids):
    """Get probability distribution for next token."""
    with torch.no_grad():
        outputs = model(input_ids)
        return outputs.logits[:, -1, :]  # Raw logits

def apply_temperature(logits, temperature):
    """Scale logits by temperature."""
    if temperature == 0:
        return logits  # Handle separately (greedy)
    return logits / temperature

# Visualize temperature effect
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
logits = get_next_token_probs(model, input_ids)

temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]
fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4), sharey=True)

for ax, temp in zip(axes, temperatures):
    scaled_logits = apply_temperature(logits, temp)
    probs = F.softmax(scaled_logits, dim=-1)[0]
    
    # Get top 15 tokens
    top_probs, top_ids = torch.topk(probs, 15)
    top_tokens = [tokenizer.decode(tid).strip() or '\\n' for tid in top_ids]
    
    ax.barh(range(len(top_tokens)), top_probs.cpu().numpy(), color='steelblue')
    ax.set_yticks(range(len(top_tokens)))
    ax.set_yticklabels(top_tokens, fontsize=7)
    ax.set_title(f'T = {temp}')
    ax.invert_yaxis()

axes[0].set_ylabel('Token')
fig.suptitle('Effect of Temperature on Next-Token Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Top-k Sampling

Top-k sampling (Fan et al., 2018) restricts sampling to only the $k$ most probable tokens, then renormalizes. This prevents sampling from the long tail of unlikely tokens.

The problem: a fixed $k$ doesn't adapt to the distribution shape. When the model is confident (one token has 95% probability), $k=50$ still includes 49 unlikely tokens. When the model is uncertain (flat distribution), $k=50$ might cut off reasonable options.

In [ ]:
def top_k_sampling(logits, k, temperature=1.0):
    """
    Top-k sampling: zero out everything except top k tokens,
    then sample from the renormalized distribution.
    """
    scaled_logits = apply_temperature(logits, temperature)
    
    # Get the k-th largest value as threshold
    top_k_values, _ = torch.topk(scaled_logits, k, dim=-1)
    threshold = top_k_values[:, -1].unsqueeze(-1)  # k-th largest
    
    # Zero out (set to -inf) everything below threshold
    filtered_logits = scaled_logits.clone()
    filtered_logits[filtered_logits < threshold] = float('-inf')
    
    # Sample from filtered distribution
    probs = F.softmax(filtered_logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    
    return next_token, probs

# Demonstrate top-k
prompt = "Once upon a time in a land far away"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
logits = get_next_token_probs(model, input_ids)

print(f"Prompt: '{prompt}'\n")
for k in [5, 20, 50, 200]:
    tokens = []
    for _ in range(10):
        token, _ = top_k_sampling(logits, k=k, temperature=1.0)
        tokens.append(tokenizer.decode(token[0]).strip())
    print(f"  k={k:3d}: {tokens}")

### 3.3 Top-p (Nucleus) Sampling

Top-p sampling (Holtzman et al., 2020) is the adaptive answer to top-k's fixed cutoff. Instead of keeping the top $k$ tokens, we keep the **smallest set of tokens whose cumulative probability exceeds $p$**.

This means:
- When the model is confident (one token at 95%), nucleus might include just 1-2 tokens
- When the model is uncertain (flat distribution), it might include 100+ tokens

The distribution itself determines how many tokens to consider.

In [ ]:
def top_p_sampling(logits, p, temperature=1.0):
    """
    Top-p (nucleus) sampling: keep smallest set of tokens
    whose cumulative probability >= p.
    """
    scaled_logits = apply_temperature(logits, temperature)
    probs = F.softmax(scaled_logits, dim=-1)
    
    # Sort probabilities in descending order
    sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
    
    # Compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # Create mask: remove tokens with cumulative prob above threshold
    # We shift right by 1 so that the token that crosses p is included
    sorted_mask = cumulative_probs - sorted_probs > p
    sorted_probs[sorted_mask] = 0.0
    
    # Renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    
    # Sample from sorted distribution
    sampled_index = torch.multinomial(sorted_probs, num_samples=1)
    next_token = sorted_indices.gather(-1, sampled_index)
    
    # Count how many tokens were in the nucleus
    nucleus_size = (~sorted_mask).sum().item()
    
    return next_token, nucleus_size

# Compare nucleus sizes across different contexts
prompts = [
    "2 + 2 =",        # Very predictable
    "The capital of France is",  # Somewhat predictable
    "I think that",   # Open-ended
    "The",            # Very open-ended
]

print("Nucleus sizes at p=0.9 for different prompts:\n")
for prompt in prompts:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    logits = get_next_token_probs(model, input_ids)
    _, nucleus_size = top_p_sampling(logits, p=0.9)
    print(f"  '{prompt}' → nucleus size: {nucleus_size} tokens")

In [ ]:
# Visualize: nucleus sampling adapts to distribution shape
fig, axes = plt.subplots(1, len(prompts), figsize=(20, 4))

for ax, prompt in zip(axes, prompts):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    logits = get_next_token_probs(model, input_ids)
    probs = F.softmax(logits, dim=-1)[0]
    
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    sorted_probs = sorted_probs[:50].cpu().numpy()  # Show top 50
    cumprobs = np.cumsum(sorted_probs)
    
    # Find nucleus boundary at p=0.9
    nucleus_end = np.searchsorted(cumprobs, 0.9) + 1
    
    colors = ['#2ecc71' if i < nucleus_end else '#ecf0f1' for i in range(len(sorted_probs))]
    ax.bar(range(len(sorted_probs)), sorted_probs, color=colors, edgecolor='none')
    ax.axvline(x=nucleus_end - 0.5, color='red', linestyle='--', alpha=0.7)
    ax.set_title(f"'{prompt}'\nnucleus={nucleus_end}", fontsize=9)
    ax.set_xlabel('Token rank')

axes[0].set_ylabel('Probability')
fig.suptitle('Nucleus Sampling Adapts to Distribution Shape (p=0.9)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Complete Sampling-Based Generator

Let's combine temperature, top-k, and top-p into a single generation function and compare the outputs.

In [ ]:
def generate_with_sampling(
    model, tokenizer, prompt, max_new_tokens=50,
    temperature=1.0, top_k=0, top_p=1.0
):
    """
    Full generation with temperature + top-k + top-p.
    top_k=0 means no top-k filtering.
    top_p=1.0 means no nucleus filtering.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)
            logits = outputs.logits[:, -1, :]
        
        # Apply temperature
        if temperature != 1.0:
            logits = logits / temperature
        
        # Apply top-k filtering
        if top_k > 0:
            top_k_values, _ = torch.topk(logits, top_k, dim=-1)
            threshold = top_k_values[:, -1].unsqueeze(-1)
            logits[logits < threshold] = float('-inf')
        
        # Apply top-p (nucleus) filtering
        if top_p < 1.0:
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            mask = (cumulative - sorted_probs) > top_p
            sorted_probs[mask] = 0.0
            # Scatter back
            probs.zero_()
            probs.scatter_(-1, sorted_indices, sorted_probs)
            probs = probs / probs.sum(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits, dim=-1)
        
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=-1)
        
        if next_token[0, 0].item() == tokenizer.eos_token_id:
            break
    
    full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return full_text[len(prompt):]

# Compare strategies
prompt = "In a world where robots have feelings,"
configs = [
    {"name": "Greedy (T=0.01)",       "temperature": 0.01, "top_k": 0,  "top_p": 1.0},
    {"name": "Low temp (T=0.3)",       "temperature": 0.3,  "top_k": 0,  "top_p": 1.0},
    {"name": "Top-k=50, T=0.7",       "temperature": 0.7,  "top_k": 50, "top_p": 1.0},
    {"name": "Nucleus p=0.9, T=0.7",  "temperature": 0.7,  "top_k": 0,  "top_p": 0.9},
    {"name": "High temp (T=1.5)",     "temperature": 1.5,  "top_k": 0,  "top_p": 1.0},
]

print(f"Prompt: '{prompt}'\n")
for cfg in configs:
    text = generate_with_sampling(
        model, tokenizer, prompt, max_new_tokens=40,
        temperature=cfg["temperature"], top_k=cfg["top_k"], top_p=cfg["top_p"]
    )
    print(f"  [{cfg['name']}]")
    print(f"  {text.strip()}\n")

---
## 4. KV Caching — The Critical Optimization

In our naive decode loop, at step $t$ we feed the **entire sequence** (prompt + all generated tokens so far) through the model. This means we recompute attention for every previous token at every step.

**KV caching** fixes this: the key ($K$) and value ($V$) projections for each attention layer are computed once per token and cached. On subsequent steps, we only compute $K$ and $V$ for the new token and append them to the cache.

The speedup is dramatic:
- Without cache: step $t$ processes $t$ tokens → total work is $O(n^2)$
- With cache: step $t$ processes 1 token (but attends to cached $t-1$ keys/values) → total work is $O(n)$ in model forward passes

Let's implement both and measure.

In [ ]:
def generate_without_cache(model, tokenizer, prompt, max_new_tokens=50):
    """Naive generation: full forward pass every step."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    
    start = time.perf_counter()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(generated)  # Process ENTIRE sequence
            logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=-1)
        if next_token[0, 0].item() == tokenizer.eos_token_id:
            break
    elapsed = time.perf_counter() - start
    
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return text, elapsed, max_new_tokens


def generate_with_cache(model, tokenizer, prompt, max_new_tokens=50):
    """Optimized generation: use KV cache."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    past_key_values = None
    
    start = time.perf_counter()
    for step in range(max_new_tokens):
        with torch.no_grad():
            if past_key_values is None:
                # First step: process entire prompt
                outputs = model(generated, use_cache=True)
            else:
                # Subsequent steps: only process the NEW token
                outputs = model(
                    generated[:, -1:],  # Just the last token!
                    past_key_values=past_key_values,
                    use_cache=True
                )
            
            logits = outputs.logits[:, -1, :]
            past_key_values = outputs.past_key_values
        
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=-1)
        if next_token[0, 0].item() == tokenizer.eos_token_id:
            break
    elapsed = time.perf_counter() - start
    
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return text, elapsed, max_new_tokens

# Warmup
_ = generate_with_cache(model, tokenizer, "warmup", max_new_tokens=5)
_ = generate_without_cache(model, tokenizer, "warmup", max_new_tokens=5)

print("KV Cache Benchmark")
print("=" * 60)

In [ ]:
# Benchmark across different generation lengths
prompt = "Artificial intelligence will transform the world because"
lengths = [10, 25, 50, 100, 150]
no_cache_times = []
cache_times = []

for n_tokens in lengths:
    # Without cache
    _, t_no_cache, _ = generate_without_cache(model, tokenizer, prompt, max_new_tokens=n_tokens)
    no_cache_times.append(t_no_cache)
    
    # With cache
    _, t_cache, _ = generate_with_cache(model, tokenizer, prompt, max_new_tokens=n_tokens)
    cache_times.append(t_cache)
    
    speedup = t_no_cache / t_cache if t_cache > 0 else float('inf')
    print(f"  {n_tokens:3d} tokens: no_cache={t_no_cache:.3f}s | cache={t_cache:.3f}s | speedup={speedup:.1f}x")

# Plot the benchmark
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(lengths, no_cache_times, 'o-', label='Without KV Cache', color='#e74c3c', linewidth=2)
ax1.plot(lengths, cache_times, 's-', label='With KV Cache', color='#2ecc71', linewidth=2)
ax1.set_xlabel('Number of generated tokens')
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Generation Time: Cache vs No Cache')
ax1.legend()
ax1.grid(True, alpha=0.3)

speedups = [nc / c if c > 0 else 0 for nc, c in zip(no_cache_times, cache_times)]
ax2.bar(range(len(lengths)), speedups, color='#3498db', edgecolor='white')
ax2.set_xticks(range(len(lengths)))
ax2.set_xticklabels([f'{l} tokens' for l in lengths])
ax2.set_ylabel('Speedup factor')
ax2.set_title('KV Cache Speedup Factor')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nThe speedup grows with sequence length because without cache,")
print(f"recomputation cost grows quadratically with sequence length.")

### What the KV cache actually stores

For a model with $L$ layers and $d$ hidden dimensions, the KV cache stores:
- Per layer: a key tensor and value tensor, each of shape `[batch, num_heads, seq_len, head_dim]`
- Total memory: $2 \times L \times n \times d$ floating point values per sequence

For a 70B parameter model generating 4096 tokens, the KV cache alone can be **several gigabytes** per request. This is why KV cache memory is the primary constraint on batch size in production serving.

In [ ]:
# Inspect the actual KV cache structure
input_ids = tokenizer.encode("Hello world", return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(input_ids, use_cache=True)

past = outputs.past_key_values
print(f"Number of layers with cached KV: {len(past)}")
print(f"Each layer stores: (key, value)")
print(f"Key shape:   {past[0][0].shape}  [batch, num_heads, seq_len, head_dim]")
print(f"Value shape: {past[0][1].shape}  [batch, num_heads, seq_len, head_dim]")

# Calculate total cache size
total_elements = sum(
    k.numel() + v.numel() for k, v in past
)
bytes_per_element = 4  # float32
total_bytes = total_elements * bytes_per_element
print(f"\nTotal KV cache for {input_ids.shape[1]} tokens: {total_bytes:,} bytes ({total_bytes/1024:.1f} KB)")
print(f"Per token: {total_bytes / input_ids.shape[1]:,.0f} bytes")

---
## 5. Stopping Criteria

Generation needs to know when to stop. There are several strategies:

1. **EOS token:** The model outputs a special end-of-sequence token
2. **Max length:** Hard limit on number of generated tokens
3. **Custom criteria:** Stop on specific strings, patterns, or conditions

In [ ]:
class StoppingCriteria:
    """Base class for stopping criteria."""
    def should_stop(self, generated_ids, generated_text):
        raise NotImplementedError

class MaxLengthStop(StoppingCriteria):
    def __init__(self, max_tokens):
        self.max_tokens = max_tokens
    def should_stop(self, generated_ids, generated_text):
        return len(generated_ids) >= self.max_tokens

class EOSStop(StoppingCriteria):
    def __init__(self, eos_token_id):
        self.eos_token_id = eos_token_id
    def should_stop(self, generated_ids, generated_text):
        return len(generated_ids) > 0 and generated_ids[-1] == self.eos_token_id

class StopStringCriteria(StoppingCriteria):
    """Stop when any of the given strings appear in output."""
    def __init__(self, stop_strings):
        self.stop_strings = stop_strings
    def should_stop(self, generated_ids, generated_text):
        for s in self.stop_strings:
            if s in generated_text:
                return True
        return False

class RepetitionStop(StoppingCriteria):
    """Stop if the last N tokens repeat a pattern."""
    def __init__(self, pattern_length=10):
        self.pattern_length = pattern_length
    def should_stop(self, generated_ids, generated_text):
        if len(generated_ids) < self.pattern_length * 2:
            return False
        recent = generated_ids[-self.pattern_length:]
        previous = generated_ids[-2*self.pattern_length:-self.pattern_length]
        return recent == previous


def generate_with_stopping(model, tokenizer, prompt, criteria_list, max_tokens=200):
    """Generate with multiple stopping criteria."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    new_token_ids = []
    past_key_values = None
    stop_reason = "max_tokens"
    
    for step in range(max_tokens):
        with torch.no_grad():
            if past_key_values is None:
                outputs = model(generated, use_cache=True)
            else:
                outputs = model(generated[:, -1:], past_key_values=past_key_values, use_cache=True)
            logits = outputs.logits[:, -1, :]
            past_key_values = outputs.past_key_values
        
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=-1)
        new_token_ids.append(next_token[0, 0].item())
        
        generated_text = tokenizer.decode(new_token_ids)
        
        for criteria in criteria_list:
            if criteria.should_stop(new_token_ids, generated_text):
                stop_reason = criteria.__class__.__name__
                return generated_text, stop_reason, step + 1
    
    return tokenizer.decode(new_token_ids), stop_reason, max_tokens

# Demo: stop on specific strings (useful for chat/instruction format)
criteria = [
    StopStringCriteria(["\n\n", ".", "!"]),
    MaxLengthStop(100),
]

prompt = "The three most important things in life are"
text, reason, steps = generate_with_stopping(model, tokenizer, prompt, criteria)
print(f"Prompt: '{prompt}'")
print(f"Generated: '{text}'")
print(f"Stopped by: {reason} after {steps} tokens")

---
## 6. Structured Output — Constrained Decoding

One of the most practically important inference-time techniques: **forcing the model to produce valid structured output** (JSON, SQL, code, etc.) by constraining which tokens can be generated at each step.

The idea: at each decoding step, we know which tokens are *valid* given the current output state. We mask out invalid tokens before sampling.

### Simple JSON Constrained Decoder

Let's build a decoder that forces valid JSON key-value pairs.

In [ ]:
import json
import re

class SimpleJSONConstraint:
    """
    A simple constrained decoder that ensures the model outputs
    a valid JSON object with specified keys.
    
    Strategy: we build the JSON character by character, using the model
    for value generation but forcing the structural tokens ourselves.
    """
    
    def __init__(self, required_keys, tokenizer):
        self.required_keys = required_keys
        self.tokenizer = tokenizer
    
    def generate_json(self, model, prompt, max_value_tokens=20):
        """
        Generate a JSON object with the required keys.
        The model fills in values; structure is enforced.
        """
        # Start building the JSON
        json_str = '{'
        
        for i, key in enumerate(self.required_keys):
            # Add key (forced)
            if i > 0:
                json_str += ', '
            json_str += f'"{key}": "'
            
            # Let model generate the value
            full_prompt = prompt + "\n" + json_str
            input_ids = self.tokenizer.encode(full_prompt, return_tensors="pt").to(device)
            
            value_tokens = []
            past_key_values = None
            
            for _ in range(max_value_tokens):
                with torch.no_grad():
                    if past_key_values is None:
                        outputs = model(input_ids, use_cache=True)
                    else:
                        outputs = model(input_ids[:, -1:], past_key_values=past_key_values, use_cache=True)
                    logits = outputs.logits[:, -1, :]
                    past_key_values = outputs.past_key_values
                
                # Mask out tokens that would break JSON
                # Block: newlines, quotes (we control when to close), backslash
                probs = F.softmax(logits / 0.7, dim=-1)
                
                next_token = torch.multinomial(probs, num_samples=1)
                next_text = self.tokenizer.decode(next_token[0])
                
                # Stop value if we see a quote or newline
                if '"' in next_text or '\n' in next_text:
                    break
                
                value_tokens.append(next_text)
                input_ids = torch.cat([input_ids, next_token], dim=-1)
            
            value = ''.join(value_tokens).strip()
            # Clean up value — remove any control chars
            value = re.sub(r'[\x00-\x1f]', '', value)
            json_str += value + '"'
        
        json_str += '}'
        return json_str

# Demo
constraint = SimpleJSONConstraint(
    required_keys=["name", "occupation", "fun_fact"],
    tokenizer=tokenizer
)

prompt = "Generate a character profile for a science fiction story."
result = constraint.generate_json(model, prompt, max_value_tokens=15)
print(f"Generated JSON:\n{result}")

# Validate it's actually valid JSON
try:
    parsed = json.loads(result)
    print(f"\nValid JSON! Parsed:")
    for k, v in parsed.items():
        print(f"  {k}: {v}")
except json.JSONDecodeError as e:
    print(f"\nJSON parse error: {e}")

### How production systems do constrained decoding

Our simple approach forces structure by controlling the generation loop. Production systems like **Outlines**, **Guidance**, and **LMQL** use more sophisticated methods:

1. **Finite State Machine (FSM) guidance:** Define a grammar (e.g., JSON schema) as an FSM. At each step, compute which tokens are valid transitions and mask all others.

2. **Grammar-constrained decoding:** Use a formal grammar (e.g., EBNF) and a parser to determine valid next tokens.

3. **Logit bias:** Instead of hard masking, add large negative values to logits of invalid tokens.

The key insight: **the model's probability distribution is still used for choosing among valid tokens.** We're not overriding the model — we're restricting its choices to structurally valid options.

In [ ]:
# Demonstrate logit bias approach
def generate_with_logit_bias(model, tokenizer, prompt, bias_dict, max_new_tokens=30):
    """
    Generate with logit biases — same mechanism the OpenAI API exposes.
    bias_dict: {token_id: bias_value}
    Positive bias = more likely, negative = less likely, -100 = effectively banned.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    past_kv = None
    
    for _ in range(max_new_tokens):
        with torch.no_grad():
            if past_kv is None:
                outputs = model(generated, use_cache=True)
            else:
                outputs = model(generated[:, -1:], past_key_values=past_kv, use_cache=True)
            logits = outputs.logits[:, -1, :]
            past_kv = outputs.past_key_values
        
        # Apply biases
        for token_id, bias in bias_dict.items():
            logits[0, token_id] += bias
        
        probs = F.softmax(logits / 0.7, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=-1)
        
        if next_token[0, 0].item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(generated[0], skip_special_tokens=True)

# Example: ban certain words by heavily penalizing their token IDs
prompt = "The best programming language is"

# Normal generation
normal = generate_with_logit_bias(model, tokenizer, prompt, {}, max_new_tokens=20)
print(f"Normal: {normal}")

# Ban "Python" token and boost "Rust" 
python_id = tokenizer.encode(" Python")[0]
rust_id = tokenizer.encode(" Rust")[0] if " Rust" in tokenizer.get_vocab() else tokenizer.encode(" C")[0]
biased = generate_with_logit_bias(
    model, tokenizer, prompt,
    {python_id: -100.0, rust_id: 5.0},
    max_new_tokens=20
)
print(f"Biased: {biased}")

---
## Summary

We've built every major component of an inference engine from scratch:

| Component | Key Insight |
|-----------|------------|
| **Autoregressive loop** | One token at a time; each depends on all previous |
| **Greedy decoding** | Fast but repetitive; doesn't find optimal sequences |
| **Beam search** | Explores multiple paths; good for translation, bad for creativity |
| **Temperature** | Controls distribution sharpness; low=focused, high=creative |
| **Top-k sampling** | Fixed cutoff; simple but doesn't adapt to confidence |
| **Top-p (nucleus)** | Adaptive cutoff; the go-to method for open-ended generation |
| **KV caching** | Cache key/value projections; avoids quadratic recomputation |
| **Stopping criteria** | EOS, max length, custom patterns, repetition detection |
| **Constrained decoding** | Force valid structure by masking invalid tokens |

**Next:** `07-inference-engine/advanced.ipynb` — speculative decoding, continuous batching, paged attention